# Setup Instructions

**If you're running this on a remote server (Jupyter/Colab/etc.), you need to upload the required files first!**

## Required Files to Upload:
1. `dataset.py`
2. `linear_region_functions.py`
3. `metrics.py`
4. `lorenz63_train.npy`
5. `lorenz63_test.npy`

## Quick Upload Method:
Run the cell below to check which files are missing, then upload them using Jupyter's file upload interface.


# Almost-Linear RNNs Tutorial

In [ ]:
# OPTIONAL: File Upload Helper (for Google Colab or similar)
# Uncomment and run this cell if you need to upload files

# For Google Colab:
# from google.colab import files
# uploaded = files.upload()  # This will prompt you to select files

# For Jupyter Notebook (if file upload widget is available):
# from ipywidgets import FileUpload
# uploader = FileUpload(accept='.py,.npy', multiple=True)
# display(uploader)

# After uploading, the files will be in the current directory
# You can verify by running: !ls -la


In [ ]:
# Setup: Check for required files and set up paths
import sys
import os

# Check which files exist in current directory
required_files = ['dataset.py', 'linear_region_functions.py', 'metrics.py', 
                  'lorenz63_train.npy', 'lorenz63_test.npy']

print("="*80)
print("Checking for required files...")
print("="*80)

missing_files = []
existing_files = []

for file in required_files:
    if os.path.exists(file):
        existing_files.append(file)
        print(f"✓ Found: {file}")
    else:
        missing_files.append(file)
        print(f"✗ Missing: {file}")

print("="*80)

if missing_files:
    print("\n⚠ WARNING: Some required files are missing!")
    print("\nIf you're running on a remote server, you need to upload these files:")
    for file in missing_files:
        print(f"  - {file}")
    print("\nTo upload files in Jupyter:")
    print("  1. Click the 'Upload' button in the file browser")
    print("  2. Select the files from your local machine")
    print("  3. Or use: from google.colab import files; files.upload()  (for Colab)")
    print("\nThe files should be in: C:\\Users\\irpay\\Downloads\\ALRNN-DSR-main\\alrnn_python\\")
    print("\n⚠ Continuing anyway - imports may fail if files are not uploaded.")
else:
    print("\n✓ All required files found!")

# Add current directory to Python path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)
    print(f"\n✓ Added current directory to Python path: {current_dir}")

print("="*80)

import numpy as np
import matplotlib.pyplot as plt
import math
from tqdm import trange

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

import torch
import torch.nn as nn
from torch.nn.init import uniform_
import torch.nn.functional as F
from torch import optim

from dataset import TimeSeriesDataset
from linear_region_functions import *
import linear_region_functions as lrf
from metrics import state_space_divergence_binning, power_spectrum_error

from matplotlib.colors import Normalize
from collections import Counter
import seaborn as sns
import copy
import pandas as pd

# ============================================================================
# GPU Setup - Automatically detect and use GPU if available
# ============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("="*80)
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("CUDA not available, using CPU")
print("="*80)

## AL-RNN Architecture

In [ ]:
class AL_RNN(nn.Module):
    def __init__(self, M, P, N):
        super(AL_RNN, self).__init__()

        # Initialize model dimensions
        self.M = M
        self.P = P
        self.N = N   

        # Initialize model parameters A, W, h, B    
        self.A, self.W, self.h = self.initialize_AWh_random()
        self.B = self.init_uniform((self.N,self.M))
        
    def forward(self, z):
        # Make a copy of the input tensor to retain unactivated values
        z_unactivated=torch.clone(z)

        # Apply ReLU activation on the last P units
        z[:, -self.P:] = F.relu(z[:, -self.P:])

        # Compute the forward pass
        return self.A * z_unactivated + z @ self.W.t() + self.h
    
    
    def initialize_AWh_random(self):
        #Randomly initialize A, W, h
        A = nn.Parameter(torch.diagonal(self.normalized_positive_definite(self.M),0)) # Create diagonal matrix A from normalized positive definite matrix
        W = nn.Parameter(torch.randn(self.M, self.M)*0.01) # Initialize weight matrix W with gaussian random numbers
        h = nn.Parameter(torch.zeros(self.M)) # Initialize bias vector h to zero
        return A, W, h
    
    
    def normalized_positive_definite(self,M):
        # Generate a normalized positive definite matrix
        R = np.random.randn(M, M).astype(np.float32)
        K = np.matmul(R.T, R) / M + np.eye(M)  # R'R ./ M + I
        eigenvalues = np.linalg.eigvals(K)
        lambda_max = np.max(np.abs(eigenvalues))
        return torch.tensor(K / lambda_max).float()
    
    
    def init_uniform(self, shape):
        # Initialize a tensor with a uniform distribution within range [-1/sqrt(M), 1/sqrt(M)]
        tensor = torch.empty(*shape)
        r = 1 / math.sqrt(shape[0])
        torch.nn.init.uniform_(tensor, -r, r)
        return nn.Parameter(tensor, requires_grad=True)


@torch.no_grad() # Disables gradient calculation to save memory and computation
def predict_free_sequence(model, x, T):
    # Predicts a sequence without updating model parameters
    b, N = x.size()

    Z = torch.empty(size=(T, b, model.M), device=x.device) # Initialize output tensor for the predicted sequence
    z = x @ model.B # Initialize first latent state
    z[:,0:N] = x

    # Predict sequence by passing previous state through the model
    for t in range(0, T):
        z = model(z)  
        Z[t] = z    
    return Z.permute(1, 0, 2)

## Training routine

In [ ]:
def predict_sequence_using_gtf(model, x, alpha, n_interleave):
    # Predicts a sequence using teacher forcing (only for training)
    x_ = x.permute(1, 0, 2) # Permute input to shape (sequence_length, batch_size, feature_dim)
    T, b, dx = x_.size() # T: sequence length, b: batch size, dx: feature dimension
    Z = torch.empty(size=(T, b, model.M), device=x.device)
    z =  x_[0] @ model.B # Initialize first latent state
    z = teacher_force(z, x_[0], alpha=1, N=model.N) # Apply teacher forcing to the initial state  25 / 11 / 2025 

    # Generate sequence predictions
    for t in range(0, T):
        # Apply teacher forcing at regular intervals
        if (t % n_interleave == 0) and (t > 0):
            z = teacher_force(z, x_[t], alpha, N=model.N)  # Apply  25 / 11 / 2025 
            
        # Update the latent state using the model
        z = model(z)
        Z[t] = z
    return Z.permute(1, 0, 2)

def teacher_force(z, x, alpha, N):
    # Teacher force the state z
    # N: data dimension (number of readout units)
    z[:, :N] = alpha * x + (1 - alpha) * z[:, :N]
    return z

def train_sh(model, dataset, optimizer, scheduler, loss_fn, num_epochs, alpha, n_interleave, lambda_l1=0.0, l1_mode='activated', batches_per_epoch=50, ssi=25, use_best_model=True): # 10/11/2025: Added l1_mode for different sparsity methods
    model.train() # Set model to training mode
    best_model = copy.deepcopy(model) # Initialize best_model saver
    losses = []
    klx = []
    dh = []
    
    with trange(num_epochs, desc="Training Progress") as epochs:
        # Loop over batches in each epoch
        for e in epochs:
            epoch_losses = []
            
            for _ in range(batches_per_epoch):
                optimizer.zero_grad() # Reset gradients for the optimizer
                
                x, y, s = dataset.sample_batch() # Sample a batch of data (x: inputs, y: targets)
                z_hat = predict_sequence_using_gtf(model, x, alpha, n_interleave) # Predict sequence using teacher forcing
                
                # 10/11/2025: Sparsity regularization term (L1 penalty)
                if l1_mode == 'activated':
                    # Original method: Penalize the activated (positive) part of the latent state
                    l1_penalty = torch.mean(torch.abs(F.relu(z_hat[:, :, -model.P:])))
                elif l1_mode == 'full':
                    # Alternative formulation: Penalize the entire latent state
                    l1_penalty = torch.mean(torch.abs(z_hat))
                else:
                    l1_penalty = 0.0
                
                # 10/11/2025: Calculate combined loss
                mse_loss = loss_fn(z_hat[:,:,:model.N], y) # Calculate loss
                loss = mse_loss + lambda_l1 * l1_penalty
                
                loss.backward() # Backward pass
                optimizer.step()
                epoch_losses.append(loss.item())
        

            scheduler.step() # Adjust learning rate based on the scheduler
            
            # Compute and store average loss for the epoch
            average_epoch_loss = sum(epoch_losses) / len(epoch_losses)
            epochs.set_postfix(loss=average_epoch_loss)
            losses.append(average_epoch_loss)

            # Save dynamical systems performance metrics Dstsp and DH each #ssi(scalar saving interval) epochs
            if e % ssi == 0:
                with torch.no_grad():
                    z_test = predict_free_sequence(model, dataset.X.clone().detach()[0:1,:], 10000) # Predict sequence using teacher forcing
                    klx.append(state_space_divergence_binning(z_test[0,:,0:model.N], dataset.X.clone().detach())) # Calculate Dstsp
                    dh.append(power_spectrum_error(z_test[0,:,0:model.N], dataset.X.clone().detach()[0:10000,:])) # Calculate DH
                    
                    if torch.argmin(torch.tensor(klx)) + 1 == len(torch.tensor(klx)): # Check if current model is best according to Dstsp
                        best_model = copy.deepcopy(model) # Copy the best model

    if use_best_model: # If true the best model (according to Dstsp) during training is returned, else the one from the last epoch
        model.load_state_dict(best_model.state_dict())
    
    return [losses, klx, dh]

## Load Data

In [ ]:
X_train = np.load("lorenz63_train.npy").astype(np.float32)[500:]
X_test = np.load("lorenz63_test.npy").astype(np.float32)[500:]
T_train, N = X_train.shape
T_test = X_test.shape[0]

print(f"Data loaded: Train shape {X_train.shape}, Test shape {X_test.shape}")
print(f"Data will be automatically moved to {device} during training")

## Initialize Model

In [ ]:
# number readout units
N=X_train.shape[-1]
#number of units in total
M=20
#number of piecewise linear units
P=2

model = AL_RNN(M=M, P=P, N=N)

## Training hyperparameters

In [ ]:
batch_size = 16
sequence_length = 200
#teacher forcing alpha (GTF)
alpha=1
#teacher forcing interval (readout units are forced every time steps)
n_interleave=16

loss_fn = nn.MSELoss()
dataset = TimeSeriesDataset(X_train, sequence_length=sequence_length, batch_size=batch_size)

## Optimization

In [ ]:
# optimization
num_epochs =10
start_learning_rate = 1e-3
optimizer = torch.optim.RAdam(model.parameters(), lr=start_learning_rate)
# Setup scheduler
end_learning_rate = 1e-5
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=np.exp(np.log(end_learning_rate/start_learning_rate)/num_epochs))

In [ ]:
# 10/11/2025: Sparsity regularization hyperparameter
lambda_l1 = 1e-4

## Training

In [ ]:
# 10/11/2025: Pass lambda_l1 and l1_mode to the training function
# l1_mode can be 'activated' (your original method) 
metrics = train_sh(model, dataset, optimizer, scheduler, loss_fn, num_epochs=num_epochs, alpha=alpha, n_interleave=n_interleave, lambda_l1=lambda_l1, l1_mode='activated', batches_per_epoch=50, ssi=20)

In [ ]:
plt.rcParams["figure.figsize"] = (7,4)
plt.rcParams.update({'font.size': 10})
fig = plt.figure()

ax = fig.add_subplot(111)
ax.plot(metrics[0],lw=3)
ax.set_title('Training loss')
ax.set_xlabel('epoch')
ax.set_ylabel('loss')
ax.set_yscale("log")

plt.tight_layout()
plt.show()

## Saving/Loading

In [ ]:
torch.save(model.state_dict(), "models/lorenz_m20_p2")

In [ ]:
model = AL_RNN(M=M, P=P, N=N)
model.load_state_dict(torch.load("models/lorenz_m20_p2"))

## Analysis

In [ ]:
X_test_torch=torch.tensor(X_test[:]).unsqueeze(0)

T_gen=10000 # Sequence length
T_r=1000 # Transient cutoff length
orbit=predict_free_sequence(model, X_test_torch[:,0,:],T_gen+T_r).detach().numpy()[0][T_r:,:] # Predict a sequence

Dstsp = state_space_divergence_binning(torch.tensor(orbit[:,0:model.N]), X_test_torch[0,:,:])
DH = power_spectrum_error(torch.tensor(orbit[:,0:model.N]), X_test_torch[0,0:T_gen,:])
print("State space distance (Dstsp): ", Dstsp)
print("Hellinger Distance (DH): ", DH)

In [ ]:
Blues = plt.cm.Blues
plt.rcParams["lines.linewidth"] = .35
plt.rcParams["figure.figsize"] = (7,5)
plt.rcParams["lines.linewidth"] = 2.
plt.rcParams.update({'font.size': 10})
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
xs = orbit[:, 0]
ys = orbit[:, 1]
zs = orbit[:, 2]

ax.plot(X_test[:T_gen, 0], X_test[:T_gen, 1], X_test[:T_gen, 2], color=Blues (0.9), label="Ground Truth")
ax.plot(xs, ys, zs, color=Blues (0.6), alpha=1., label="Freely Generated")

plt.legend(loc="upper left")

plt.axis("off")
plt.show()

In [ ]:
generated_latent=orbit[:,-P:] #latent sequence in PWL units
generated_observations=orbit[:,:N] #predicted readout sequence 

bits = lrf.convert_to_bits(generated_latent) #sequence of current active subregion, encoded in bits
regions, unique_regions = lrf.unique_regions_crossed(bits, M) #Number of linear subregions, used linear subregions encoded as bits
frequencies=lrf.frequency_of_regions(bits,unique_regions) #How often is each subregion visited

#Compute order of most freqnent visited subregions
frequency_list = np.copy(frequencies)
regions_list = np.copy(unique_regions)    
most_frequent_regions = []
for _ in range(len(frequency_list)):
    index = np.argmax(frequency_list)
    most_frequent_regions.append(regions_list[index])
    frequency_list = np.delete(frequency_list, index)
    regions_list = np.delete(regions_list, index, axis=0)

In [ ]:
connectome = lrf.connectome_with_self_connections(bits, most_frequent_regions, len(most_frequent_regions))
print("Used subregions: ",len(connectome))
print("Timesteps in subregion: ",frequencies)

In [ ]:
plt.figure(figsize=(7, 5))
sns.set_context('talk', font_scale=1.2) 
sns.set_style('white')

# Define a custom diverging colormap with higher contrast
#cmap = sns.diverging_palette(220, 20, as_cmap=True)
cmap = plt.get_cmap('Blues')

# Plot the heatmap with selected improvements
ax = sns.heatmap(data=connectome[:, :], annot=False, linewidths=0, cmap=cmap, square=True,
                 yticklabels=False, xticklabels=False, cbar_kws={'label': 'Transition Probability'})

# Rotate the colorbar label for better visibility
cbar = ax.collections[0].colorbar
cbar.set_label('Transition Probability', rotation=270, labelpad=35)

# Optionally, adjust the limits of the color map to enhance contrast
ax.collections[0].set_clim(0, np.max(connectome)-0.015)

plt.show()

## Experiment Framework

Helper functions for running systematic experiments with different hyperparameter configurations.


In [ ]:
def compute_analysis_metrics(model, X_test, T_gen=10000, T_r=1000, device=None):
    """
    Compute analysis metrics for a trained model:
    - D_stsp: State space divergence
    - D_H: Power spectrum error (Hellinger distance)
    - n_active_regions: Number of active linear subregions
    - region_frequencies: Frequency distribution of regions
    
    Parameters:
    -----------
    model : AL_RNN
        Trained model
    X_test : numpy array or torch tensor
        Test data of shape (T, N)
    T_gen : int
        Length of generated sequence
    T_r : int
        Transient cutoff length
    device : torch.device
        Device to use (defaults to model's device)
        
    Returns:
    --------
    dict : Dictionary containing all metrics
    """
    if device is None:
        device = next(model.parameters()).device
    
    X_test_torch = torch.tensor(X_test[:], device=device).unsqueeze(0) if isinstance(X_test, np.ndarray) else X_test.unsqueeze(0).to(device)
    
    # Generate free sequence
    orbit = predict_free_sequence(model, X_test_torch[:, 0, :], T_gen + T_r).detach().numpy()[0][T_r:, :]
    
    # Compute performance metrics
    D_stsp = state_space_divergence_binning(
        torch.tensor(orbit[:, 0:model.N]), 
        X_test_torch[0, :, :]
    )
    D_H = power_spectrum_error(
        torch.tensor(orbit[:, 0:model.N]), 
        X_test_torch[0, 0:T_gen, :]
    )
    
    # Compute linear region analysis
    generated_latent = orbit[:, -model.P:]  # PWL units
    bits = lrf.convert_to_bits(generated_latent)
    n_regions, unique_regions = lrf.unique_regions_crossed(bits, model.P)
    frequencies = lrf.frequency_of_regions(bits, unique_regions)
    
    return {
        'D_stsp': D_stsp,
        'D_H': D_H,
        'n_active_regions': n_regions,
        'region_frequencies': frequencies,
        'unique_regions': unique_regions,
        'bits': bits
    }


In [ ]:
def run_single_experiment(
    M, P, N, X_train, X_test,
    lambda_l1=0.0, l1_mode='activated',
    num_epochs=1000, batches_per_epoch=50, ssi=25,
    batch_size=16, sequence_length=200,
    alpha=1, n_interleave=16,
    start_learning_rate=1e-3, end_learning_rate=1e-5,
    use_best_model=True,
    seed=None,
    verbose=True
):
    """
    Run a single training experiment with specified hyperparameters.
    
    Parameters:
    -----------
    M : int
        Latent dimension
    P : int
        Number of PWL units
    N : int
        Data dimension
    X_train : numpy array
        Training data of shape (T, N)
    X_test : numpy array
        Test data of shape (T, N)
    lambda_l1 : float
        L1 regularization strength
    l1_mode : str
        'activated' (L1 on activated PWL units) or 'full' (L1 on entire latent)
    num_epochs : int
        Number of training epochs
    batches_per_epoch : int
        Number of batches per epoch
    ssi : int
        Scalar saving interval (evaluate metrics every ssi epochs)
    batch_size : int
        Batch size
    sequence_length : int
        Sequence length for training
    alpha : float
        Teacher forcing alpha
    n_interleave : int
        Teacher forcing interval
    start_learning_rate : float
        Initial learning rate
    end_learning_rate : float
        Final learning rate
    use_best_model : bool
        Whether to use best model (by D_stsp) or final model
    seed : int or None
        Random seed for reproducibility
    verbose : bool
        Whether to print progress
        
    Returns:
    --------
    dict : Dictionary containing:
        - model: Trained model
        - training_metrics: [losses, klx, dh] from training
        - analysis_metrics: Results from compute_analysis_metrics
        - config: Dictionary of hyperparameters used
    """
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    
    # Get device (use global device if available, else auto-detect)
    if 'device' in globals():
        use_device = device
    else:
        use_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Initialize model and move to GPU
    model = AL_RNN(M=M, P=P, N=N)
    model = model.to(use_device)
    
    # Setup dataset with device
    dataset = TimeSeriesDataset(X_train, sequence_length=sequence_length, batch_size=batch_size, device=use_device)
    
    # Setup optimizer and scheduler
    optimizer = torch.optim.RAdam(model.parameters(), lr=start_learning_rate)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(
        optimizer, 
        gamma=np.exp(np.log(end_learning_rate / start_learning_rate) / num_epochs)
    )
    
    # Setup loss function
    loss_fn = nn.MSELoss()
    
    # Train model
    if verbose:
        print(f"Training: M={M}, P={P}, lambda_l1={lambda_l1}, l1_mode={l1_mode}")
    
    training_metrics = train_sh(
        model=model,
        dataset=dataset,
        optimizer=optimizer,
        scheduler=scheduler,
        loss_fn=loss_fn,
        num_epochs=num_epochs,
        alpha=alpha,
        n_interleave=n_interleave,
        lambda_l1=lambda_l1,
        l1_mode=l1_mode,
        batches_per_epoch=batches_per_epoch,
        ssi=ssi,
        use_best_model=use_best_model
    )
    
    # Compute analysis metrics
    if verbose:
        print(f"Computing analysis metrics...")
    analysis_metrics = compute_analysis_metrics(model, X_test, device=use_device)
    
    # Store configuration
    config = {
        'M': M,
        'P': P,
        'N': N,
        'lambda_l1': lambda_l1,
        'l1_mode': l1_mode,
        'num_epochs': num_epochs,
        'batches_per_epoch': batches_per_epoch,
        'ssi': ssi,
        'batch_size': batch_size,
        'sequence_length': sequence_length,
        'alpha': alpha,
        'n_interleave': n_interleave,
        'start_learning_rate': start_learning_rate,
        'end_learning_rate': end_learning_rate,
        'use_best_model': use_best_model,
        'seed': seed
    }
    
    return {
        'model': model,
        'training_metrics': training_metrics,
        'analysis_metrics': analysis_metrics,
        'config': config
    }


In [ ]:
def run_experiment_grid(
    M_values, P, N, X_train, X_test,
    sparsity_configs,
    num_epochs=1000, batches_per_epoch=50, ssi=25,
    batch_size=16, sequence_length=200,
    alpha=1, n_interleave=16,
    start_learning_rate=1e-3, end_learning_rate=1e-5,
    use_best_model=True,
    seed=None,
    save_models=False,
    model_save_dir="models"
):
    """
    Run a grid of experiments across different M values and sparsity configurations.
    
    Parameters:
    -----------
    M_values : list of int
        List of latent dimensions to test
    P : int
        Number of PWL units
    N : int
        Data dimension
    X_train : numpy array
        Training data
    X_test : numpy array
        Test data
    sparsity_configs : list of tuples
        Each tuple is (mode_name, lambda_l1, l1_mode)
        e.g., [("none", 0.0, "activated"), ("activated", 1e-4, "activated"), ("full", 1e-4, "full")]
    num_epochs : int
        Number of training epochs
    batches_per_epoch : int
        Number of batches per epoch
    ssi : int
        Scalar saving interval
    batch_size : int
        Batch size
    sequence_length : int
        Sequence length
    alpha : float
        Teacher forcing alpha
    n_interleave : int
        Teacher forcing interval
    start_learning_rate : float
        Initial learning rate
    end_learning_rate : float
        Final learning rate
    use_best_model : bool
        Whether to use best model
    seed : int or None
        Random seed
    save_models : bool
        Whether to save trained models
    model_save_dir : str
        Directory to save models
        
    Returns:
    --------
    pandas.DataFrame : Results dataframe with columns:
        - M, P, N
        - mode_name, lambda_l1, l1_mode
        - best_D_stsp, best_D_H (best values during training)
        - final_D_stsp, final_D_H (from analysis_metrics)
        - mean_training_loss
        - n_active_regions
        - experiment_id
    list : List of experiment result dictionaries (full results)
    """
    results = []
    experiment_id = 0
    
    # Create model directory if needed
    if save_models:
        os.makedirs(model_save_dir, exist_ok=True)
    
    total_experiments = len(M_values) * len(sparsity_configs)
    print(f"Running {total_experiments} experiments...")
    print("=" * 80)
    
    for M in M_values:
        for mode_name, lambda_l1, l1_mode in sparsity_configs:
            experiment_id += 1
            print(f"\n[{experiment_id}/{total_experiments}] M={M}, mode={mode_name}, lambda_l1={lambda_l1}, l1_mode={l1_mode}")
            print("-" * 80)
            
            try:
                # Run experiment
                result = run_single_experiment(
                    M=M, P=P, N=N,
                    X_train=X_train, X_test=X_test,
                    lambda_l1=lambda_l1, l1_mode=l1_mode,
                    num_epochs=num_epochs,
                    batches_per_epoch=batches_per_epoch,
                    ssi=ssi,
                    batch_size=batch_size,
                    sequence_length=sequence_length,
                    alpha=alpha,
                    n_interleave=n_interleave,
                    start_learning_rate=start_learning_rate,
                    end_learning_rate=end_learning_rate,
                    use_best_model=use_best_model,
                    seed=seed,
                    verbose=False
                )
                
                # Extract metrics
                losses, klx, dh = result['training_metrics']
                analysis = result['analysis_metrics']
                
                # Get best metrics during training
                best_D_stsp = min(klx) if klx else np.nan
                best_D_H = min(dh) if dh else np.nan
                best_epoch_idx = np.argmin(klx) if klx else -1
                
                # Get final metrics
                final_D_stsp = analysis['D_stsp']
                final_D_H = analysis['D_H']
                
                # Mean training loss (from last 10% of epochs)
                mean_training_loss = np.mean(losses[-len(losses)//10:]) if losses else np.nan
                
                # Number of active regions
                n_active_regions = analysis['n_active_regions']
                
                # Save model if requested
                model_path = None
                if save_models:
                    model_path = os.path.join(
                        model_save_dir, 
                        f"lorenz_M{M}_P{P}_{mode_name}_lambda{lambda_l1:.0e}.pt"
                    )
                    torch.save(result['model'].state_dict(), model_path)
                
                # Store results
                results.append({
                    'experiment_id': experiment_id,
                    'M': M,
                    'P': P,
                    'N': N,
                    'mode_name': mode_name,
                    'lambda_l1': lambda_l1,
                    'l1_mode': l1_mode,
                    'num_epochs': num_epochs,
                    'best_D_stsp': best_D_stsp,
                    'best_D_H': best_D_H,
                    'best_epoch': best_epoch_idx * ssi if best_epoch_idx >= 0 else np.nan,
                    'final_D_stsp': final_D_stsp,
                    'final_D_H': final_D_H,
                    'mean_training_loss': mean_training_loss,
                    'n_active_regions': n_active_regions,
                    'model_path': model_path,
                    'full_result': result  # Store full result for detailed analysis
                })
                
                print(f"✓ Completed: best_D_stsp={best_D_stsp:.4f}, final_D_stsp={final_D_stsp:.4f}, "
                      f"n_active_regions={n_active_regions}")
                
            except Exception as e:
                print(f"✗ Failed: {str(e)}")
                results.append({
                    'experiment_id': experiment_id,
                    'M': M,
                    'P': P,
                    'N': N,
                    'mode_name': mode_name,
                    'lambda_l1': lambda_l1,
                    'l1_mode': l1_mode,
                    'error': str(e)
                })
    
    # Convert to DataFrame
    df_results = pd.DataFrame(results)
    
    print("\n" + "=" * 80)
    print("All experiments completed!")
    print("=" * 80)
    
    return df_results, results


## Quick Test Run (Start Here!)

Run this first to test with a small configuration (fast, ~2-5 minutes):


In [ ]:
# QUICK TEST: Run a small experiment to verify everything works
# This will test M=20 with no sparsity, using only 50 epochs (fast!)

print("="*80)
print("QUICK TEST RUN - Verifying setup works...")
print("="*80)

# Test with just one M value and one config, fewer epochs
test_result = run_single_experiment(
    M=20,                    # Latent dimension
    P=2,                     # Number of PWL units
    N=N,                     # Data dimension
    X_train=X_train,         # Training data
    X_test=X_test,           # Test data
    lambda_l1=0.0,           # No sparsity for quick test
    l1_mode='activated',
    num_epochs=50,           # Just 50 epochs for quick test
    batches_per_epoch=50,
    ssi=10,                  # Evaluate every 10 epochs
    batch_size=16,
    sequence_length=200,
    alpha=1,
    n_interleave=16,
    start_learning_rate=1e-3,
    end_learning_rate=1e-5,
    use_best_model=True,
    seed=42,
    verbose=True
)

# Display results
print("\n" + "="*80)
print("TEST RESULTS:")
print("="*80)
analysis = test_result['analysis_metrics']
print(f"D_stsp (state space divergence): {analysis['D_stsp']:.4f}")
print(f"D_H (power spectrum error): {analysis['D_H']:.4f}")
print(f"Number of active regions: {analysis['n_active_regions']}")

print("\n✓ Test completed successfully! You can now run the full experiment grid below.")


In [ ]:
# ============================================================================
# STEP 1: Define which M values you want to test
# ============================================================================
# Change this list to include the M values you want to experiment with
M_values = [10, 20, 40, 80]  # <-- EDIT THIS: Add/remove M values here

# ============================================================================
# STEP 2: Define sparsity configurations to test
# ============================================================================
# Each tuple is: (mode_name, lambda_l1, l1_mode)
# - mode_name: A descriptive name (e.g., "none", "activated", "full")
# - lambda_l1: L1 regularization strength (0.0 = no sparsity)
# - l1_mode: "activated" (L1 on PWL units) or "full" (L1 on entire latent)
sparsity_configs = [
    ("none", 0.0, "activated"),      # No sparsity regularization
    ("activated", 1e-4, "activated"), # L1 on activated PWL units only
    ("full", 1e-4, "full"),           # L1 on entire latent state
]

# ============================================================================
# STEP 3: Set other hyperparameters
# ============================================================================
P = 2  # Number of PWL units (usually keep this fixed)
num_epochs = 1000  # Training epochs (increase to 2000 for best configs)

# ============================================================================
# STEP 4: Run all experiments
# ============================================================================
# This will run experiments for ALL combinations of M_values and sparsity_configs
# Total experiments = len(M_values) × len(sparsity_configs)
# Example: 4 M values × 3 configs = 12 experiments

df_results, full_results = run_experiment_grid(
    M_values=M_values,              # List of M values to test
    P=P,                            # Number of PWL units
    N=N,                            # Data dimension (from data loading section)
    X_train=X_train,                # Training data (from data loading section)
    X_test=X_test,                  # Test data (from data loading section)
    sparsity_configs=sparsity_configs,  # Sparsity configurations
    num_epochs=num_epochs,          # Training epochs
    batches_per_epoch=50,           # Batches per epoch
    ssi=25,                         # Evaluation interval (every ssi epochs)
    batch_size=16,                   # Batch size
    sequence_length=200,            # Sequence length
    alpha=1,                        # Teacher forcing alpha
    n_interleave=16,                # Teacher forcing interval
    start_learning_rate=1e-3,       # Initial learning rate
    end_learning_rate=1e-5,          # Final learning rate
    use_best_model=True,            # Use best model (by D_stsp) or final model
    seed=42,                        # Random seed for reproducibility
    save_models=True,                # Save trained models to disk
    model_save_dir="models"         # Directory to save models
)

# ============================================================================
# STEP 5: View and save results
# ============================================================================
# Display summary of results
print("\n" + "="*80)
print("EXPERIMENT RESULTS SUMMARY")
print("="*80)
print(df_results[['M', 'mode_name', 'lambda_l1', 'best_D_stsp', 'final_D_stsp', 
                  'n_active_regions', 'mean_training_loss']].to_string(index=False))

# Save results to CSV file
df_results.to_csv('experiment_results.csv', index=False)
print("\n✓ Results saved to 'experiment_results.csv'")
print(f"✓ Total experiments completed: {len(df_results)}")


In [ ]:
# Utility function to load saved results
def load_experiment_results(csv_path='experiment_results.csv'):
    """
    Load experiment results from a CSV file.
    
    Parameters:
    -----------
    csv_path : str
        Path to the CSV file containing results
        
    Returns:
    --------
    pandas.DataFrame : Loaded results
    """
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} experiment results from {csv_path}")
    return df

# Utility function to find best configurations
def find_best_configurations(df_results, metric='final_D_stsp', top_k=5):
    """
    Find the top-k best configurations according to a metric.
    
    Parameters:
    -----------
    df_results : pandas.DataFrame
        Results dataframe
    metric : str
        Metric to rank by (lower is better for D_stsp and D_H)
    top_k : int
        Number of top configurations to return
        
    Returns:
    --------
    pandas.DataFrame : Top-k configurations
    """
    df_sorted = df_results.dropna(subset=[metric]).sort_values(by=metric).head(top_k)
    print(f"\nTop {top_k} configurations by {metric}:")
    print("=" * 80)
    print(df_sorted[['M', 'mode_name', 'lambda_l1', 'l1_mode', metric, 'n_active_regions']].to_string(index=False))
    return df_sorted

# Example usage:
# df_results = load_experiment_results('experiment_results.csv')
# best_configs = find_best_configurations(df_results, metric='final_D_stsp', top_k=5)


## Quick Reference: Running a Single Experiment

If you only want to test ONE specific configuration (one M value, one sparsity setting):


In [ ]:
# Example: Run a single experiment with M=20, no sparsity
# Uncomment and modify as needed:

# result = run_single_experiment(
#     M=20,                    # Latent dimension
#     P=2,                     # Number of PWL units
#     N=N,                     # Data dimension (from data loading)
#     X_train=X_train,         # Training data
#     X_test=X_test,           # Test data
#     lambda_l1=0.0,           # No sparsity
#     l1_mode='activated',     # Mode (doesn't matter when lambda_l1=0)
#     num_epochs=1000,         # Training epochs
#     seed=42,                 # Random seed
#     verbose=True             # Print progress
# )
# 
# # Access results
# model = result['model']
# training_metrics = result['training_metrics']  # [losses, klx, dh]
# analysis_metrics = result['analysis_metrics']  # D_stsp, D_H, n_active_regions, etc.
# 
# print(f"D_stsp: {analysis_metrics['D_stsp']:.4f}")
# print(f"D_H: {analysis_metrics['D_H']:.4f}")
# print(f"Active regions: {analysis_metrics['n_active_regions']}")


## Loading Previously Saved Results

If you've already run experiments and saved results to CSV:


In [ ]:
# Load previously saved results
# df_results = load_experiment_results('experiment_results.csv')
# 
# # View results
# print(df_results[['M', 'mode_name', 'lambda_l1', 'final_D_stsp', 'n_active_regions']])
# 
# # Find best configurations
# best_configs = find_best_configurations(df_results, metric='final_D_stsp', top_k=5)
# 
# # Plot results
# plot_experiment_results(df_results, metric='final_D_stsp')


In [ ]:
# Helper function to visualize results
def plot_experiment_results(df_results, metric='final_D_stsp', save_path=None):
    """
    Plot experiment results comparing different configurations.
    
    Parameters:
    -----------
    df_results : pandas.DataFrame
        Results dataframe from run_experiment_grid
    metric : str
        Metric to plot ('final_D_stsp', 'final_D_H', 'n_active_regions', etc.)
    save_path : str or None
        Path to save figure
    """
    if metric not in df_results.columns:
        print(f"Metric '{metric}' not found in results.")
        return
    
    # Filter out failed experiments
    df_plot = df_results.dropna(subset=[metric])
    
    # Create pivot table for heatmap
    pivot_data = df_plot.pivot_table(
        values=metric,
        index='M',
        columns='mode_name',
        aggfunc='mean'
    )
    
    # Plot
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot_data, annot=True, fmt='.4f', cmap='viridis', cbar_kws={'label': metric})
    plt.title(f'{metric} by M and Sparsity Mode')
    plt.xlabel('Sparsity Mode')
    plt.ylabel('Latent Dimension (M)')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
    
    plt.show()
    
    # Also create line plot
    plt.figure(figsize=(10, 6))
    for mode in df_plot['mode_name'].unique():
        mode_data = df_plot[df_plot['mode_name'] == mode]
        plt.plot(mode_data['M'], mode_data[metric], marker='o', label=mode, linewidth=2, markersize=8)
    
    plt.xlabel('Latent Dimension (M)', fontsize=12)
    plt.ylabel(metric, fontsize=12)
    plt.title(f'{metric} vs Latent Dimension', fontsize=14)
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    if save_path:
        base_path = save_path.rsplit('.', 1)[0]
        plt.savefig(f'{base_path}_line.png', dpi=300)
    
    plt.show()

# Example usage (uncomment after running experiments):
# plot_experiment_results(df_results, metric='final_D_stsp')
# plot_experiment_results(df_results, metric='n_active_regions')


In [ ]:
sns.set_context('talk', font_scale=1.0) 
plt.rcParams["lines.linewidth"] = 2.

observations_plot = generated_observations[:, :3]
bitcode_plot = lrf.convert_to_bits(generated_latent)

# Convert bitcode arrays to strings
bitcodes_str = [''.join(map(str, map(int, b))) for b in bitcode_plot]

# Count the frequency of each bitcode
bitcode_freq = Counter(bitcodes_str)
most_frequent_regions = [item[0] for item in bitcode_freq.most_common()]
frequency_order_map = {code: idx for idx, code in enumerate(most_frequent_regions)}
order = [frequency_order_map[code] for code in bitcodes_str]

# Normalize order indices for color mapping
order_array = np.array(order)
norm = Normalize(vmin=order_array.min(), vmax=order_array.max())
cmap = plt.get_cmap('cividis')
colors = cmap(norm(order_array))

fig = plt.figure(figsize=(8, 5))
#fig.tight_layout()
ax = fig.add_subplot(121, projection='3d')
scatter = ax.scatter(observations_plot[:, 0], observations_plot[:, 1], observations_plot[:, 2], c=colors, s=220)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.axis("off")

ax = fig.add_subplot(322)
for i in range(1000):
    ax.plot([i, i+1], [observations_plot[i, 0], observations_plot[i+1, 0]], color=colors[i+1])
ax.set_ylabel('X')

ax = fig.add_subplot(324)
for i in range(1000):
    ax.plot([i, i+1], [observations_plot[i, 1], observations_plot[i+1, 1]], color=colors[i+1])
ax.set_ylabel('Y')

ax = fig.add_subplot(326)
for i in range(1000):
    ax.plot([i, i+1], [observations_plot[i, 2], observations_plot[i+1, 2]], color=colors[i+1])
ax.set_ylabel('Z')

plt.tight_layout()
plt.show()